# Remote VS Code Setup (Colab / Kaggle)

Connect your local VS Code to Colab's or Kaggle's GPU and control everything from your machine.

## How It Works
1. This notebook starts a **VS Code Tunnel** on Colab/Kaggle
2. You open the tunnel URL in your local VS Code
3. You get a full VS Code editor + terminal connected to the remote GPU
4. Run scripts, edit code, debug — all from your local VS Code

## Prerequisites
- A **GitHub account** (for tunnel authentication)
- VS Code installed locally
- Colab or Kaggle notebook with GPU enabled

## Step 1: Detect Environment

In [ ]:
import os
import subprocess

# Detect if we're on Colab or Kaggle
IS_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')
IS_KAGGLE = os.path.exists('/kaggle')

if IS_COLAB:
    ENV = 'COLAB'
    WORK_DIR = '/content'
    print('Running on Google Colab')
elif IS_KAGGLE:
    ENV = 'KAGGLE'
    WORK_DIR = '/kaggle/working'
    print('Running on Kaggle')
else:
    ENV = 'LOCAL'
    WORK_DIR = os.getcwd()
    print('Running locally')

# Check GPU
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB)')
else:
    print('No GPU — enable it in Runtime settings!')

## Step 2: Install VS Code CLI (code-tunnel)

In [ ]:
%%bash
# Download and install VS Code CLI
if [ ! -f /usr/local/bin/code ]; then
    echo "Installing VS Code CLI..."
    curl -sL "https://code.visualstudio.com/sha/download?build=stable&os=cli-alpine-x64" -o /tmp/vscode_cli.tar.gz
    tar -xzf /tmp/vscode_cli.tar.gz -C /tmp/
    mv /tmp/code /usr/local/bin/code
    chmod +x /usr/local/bin/code
    rm /tmp/vscode_cli.tar.gz
    echo "VS Code CLI installed!"
else
    echo "VS Code CLI already installed"
fi
code --version

## Step 3: Upload Your Project

**Option A (recommended):** Upload a zip of your project from your local machine:
```bash
# Run on your local machine:
cd /Users/harsh/major_project
zip -r medicalq-a_rag.zip medicalq-a_rag/ \
    -x 'medicalq-a_rag/venv/*' \
    -x 'medicalq-a_rag/chroma_db/*' \
    -x 'medicalq-a_rag/chroma_pubmedqa_only/*' \
    -x 'medicalq-a_rag/node_modules/*' \
    -x 'medicalq-a_rag/.git/*' \
    -x 'medicalq-a_rag/venv/*'
```
Then upload the zip in the next cell.

**Option B:** Clone from GitHub (if you've pushed your repo).

In [ ]:
import os

PROJECT_DIR = f'{WORK_DIR}/medicalq-a_rag'

if not os.path.exists(PROJECT_DIR):
    # ---- OPTION A: Upload zip ----
    if IS_COLAB:
        from google.colab import files
        print('Upload your medicalq-a_rag.zip:')
        uploaded = files.upload()
        !unzip -q medicalq-a_rag.zip -d {WORK_DIR}
    elif IS_KAGGLE:
        print('On Kaggle: Add your dataset or upload files via the sidebar')
    
    # ---- OPTION B: Clone from GitHub ----
    # !git clone https://github.com/YOUR_USERNAME/medicalq-a_rag.git {PROJECT_DIR}
else:
    print(f'Project already exists at {PROJECT_DIR}')

os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')
!ls -la

## Step 4: Install Project Dependencies

In [ ]:
!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm -q
print('Dependencies installed!')

In [ ]:
# Set API keys
import os

if IS_COLAB:
    try:
        from google.colab import userdata
        os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
        os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
        print('API keys loaded from Colab Secrets')
    except Exception:
        print('Add API keys to Colab Secrets (key icon in sidebar)')
elif IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        os.environ['GEMINI_API_KEY'] = secrets.get_secret('GEMINI_API_KEY')
        os.environ['GROQ_API_KEY'] = secrets.get_secret('GROQ_API_KEY')
        print('API keys loaded from Kaggle Secrets')
    except Exception:
        print('Add API keys to Kaggle Secrets')

os.environ['LLM_PROVIDER'] = 'auto'
os.environ['LLM_FALLBACK_CHAIN'] = 'gemini,groq'

# Write .env file for subprocess scripts
with open('.env', 'w') as f:
    f.write(f"GEMINI_API_KEY={os.environ.get('GEMINI_API_KEY', '')}\n")
    f.write(f"GROQ_API_KEY={os.environ.get('GROQ_API_KEY', '')}\n")
    f.write(f"LLM_PROVIDER=auto\n")
    f.write(f"LLM_FALLBACK_CHAIN=gemini,groq\n")
print('.env file written')

## Step 5: Start VS Code Tunnel

**This is the main step!** Running the cell below will:
1. Start a VS Code tunnel
2. Give you a GitHub login URL — open it and authorize
3. Once authorized, you'll get a `vscode.dev` URL
4. Open that URL in your browser OR connect from your local VS Code

### To connect from local VS Code:
1. Install the **"Remote - Tunnels"** extension in VS Code
2. Press `Ctrl+Shift+P` → "Remote Tunnels: Connect to Tunnel"
3. Sign in with the same GitHub account
4. Select the tunnel name (default: `mega-rag-colab` or `mega-rag-kaggle`)

**IMPORTANT:** Keep this cell running — the tunnel stays alive as long as the cell is active.

In [ ]:
# Start VS Code tunnel
# This will print a GitHub auth URL — open it and authorize

tunnel_name = f'mega-rag-{ENV.lower()}'
print(f'Starting VS Code tunnel: {tunnel_name}')
print(f'Working directory: {os.getcwd()}')
print()
print('INSTRUCTIONS:')
print('1. A GitHub login URL will appear below — open it and authorize')
print('2. Once authorized, a vscode.dev URL will appear')
print('3. Open it in browser OR connect from local VS Code:')
print('   Ctrl+Shift+P → "Remote Tunnels: Connect to Tunnel"')
print()
print('Keep this cell running! The tunnel dies when you stop it.')
print('='*60)

!code tunnel --name {tunnel_name} --accept-server-license-terms

## Alternative: SSH Access (if tunnel doesn't work)

Use `colab-ssh` for direct SSH access from VS Code.

In [ ]:
# # Alternative: SSH via cloudflared (Colab only)
# !pip install colab-ssh -q
# from colab_ssh import launch_ssh_cloudflared
# launch_ssh_cloudflared(password='mega-rag-2026')
#
# # Then in your local terminal:
# # 1. Install cloudflared: brew install cloudflare/cloudflare/cloudflared
# # 2. Copy the SSH command printed above
# # 3. In VS Code: Remote-SSH → Connect to Host → paste the command